# KSSD2025: Kidney Stone Segmentation with Modified U-Net Models

**Paper:** *A New Annotated Dataset for Automatic Kidney Stone Segmentation and Evaluation With Modified U-Net Based Deep Learning Models* (IEEE Access, 2025)

This notebook replicates the methodology from the KSSD2025 paper, implementing and evaluating four architectures:
1. **U-Net** (baseline)
2. **U-Net++** (nested dense skip connections)
3. **U-Net3+** (full-scale skip connections)
4. **TransUNet** (Transformer + U-Net hybrid)

## 0. Install Dependencies

In [ ]:
# Uncomment and run if packages are not installed
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install numpy matplotlib Pillow scikit-learn tqdm

## 1. Imports & Configuration

In [ ]:
import os
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from sklearn.model_selection import KFold
from tqdm import tqdm
import math

warnings.filterwarnings('ignore')

# ─── Configuration ───
CONFIG = {
    'data_dir': r'c:\Users\PARDHEEV\B Tech\7th Sem\Projects\Medical-Image\kidney-stone-segmentation\data',
    'img_size': 256,          # Resize images to 256x256
    'batch_size': 16,
    'num_epochs': 150,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'n_folds': 5,
    'early_stop_patience': 20,
    'lr_patience': 10,
    'lr_factor': 0.5,
    'seed': 42,
}

# ─── Reproducibility ───
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG['seed'])

# ─── Device ───
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Dataset Class

In [ ]:
class KidneyStoneDataset(Dataset):
    """
    Custom Dataset for KSSD2025 kidney stone segmentation.
    Loads TIF images (grayscale) and corresponding binary masks.
    Applies augmentation for training and basic preprocessing for validation.
    """

    def __init__(self, image_paths, mask_paths, img_size=256, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def _apply_augmentations(self, image, mask):
        """Apply identical spatial transforms to image and mask."""
        # Random Horizontal Flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        # Random Vertical Flip
        if random.random() > 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)

        # Random Rotation (±15 degrees)
        if random.random() > 0.5:
            angle = random.uniform(-15, 15)
            image = TF.rotate(image, angle, interpolation=T.InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=T.InterpolationMode.NEAREST)

        # Random Affine (scale + translate)
        if random.random() > 0.5:
            angle = 0
            translate = [int(random.uniform(-0.1, 0.1) * self.img_size),
                         int(random.uniform(-0.1, 0.1) * self.img_size)]
            scale = random.uniform(0.9, 1.1)
            shear = 0
            image = TF.affine(image, angle, translate, scale, shear,
                              interpolation=T.InterpolationMode.BILINEAR)
            mask = TF.affine(mask, angle, translate, scale, shear,
                             interpolation=T.InterpolationMode.NEAREST)

        # Random Brightness/Contrast (image only)
        if random.random() > 0.5:
            brightness_factor = random.uniform(0.8, 1.2)
            image = TF.adjust_brightness(image, brightness_factor)

        if random.random() > 0.5:
            contrast_factor = random.uniform(0.8, 1.2)
            image = TF.adjust_contrast(image, contrast_factor)

        return image, mask

    def __getitem__(self, idx):
        # Load image (grayscale) and mask
        image = Image.open(self.image_paths[idx]).convert('L')
        mask = Image.open(self.mask_paths[idx]).convert('L')

        # Resize
        image = TF.resize(image, [self.img_size, self.img_size],
                          interpolation=T.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.img_size, self.img_size],
                         interpolation=T.InterpolationMode.NEAREST)

        # Apply augmentations (training only)
        if self.augment:
            image, mask = self._apply_augmentations(image, mask)

        # Convert to tensor and normalize
        image = TF.to_tensor(image)  # [1, H, W], values in [0, 1]
        mask = TF.to_tensor(mask)    # [1, H, W]

        # Binarize mask: any pixel > 0 becomes 1
        mask = (mask > 0).float()

        return image, mask

## 3. Data Loading & Visualization

In [ ]:
# ─── Load file paths ───
image_dir = os.path.join(CONFIG['data_dir'], 'image')
label_dir = os.path.join(CONFIG['data_dir'], 'label')

# Get sorted file lists and match image-mask pairs by filename
image_files = sorted(os.listdir(image_dir), key=lambda x: int(os.path.splitext(x)[0]))
label_files = sorted(os.listdir(label_dir), key=lambda x: int(os.path.splitext(x)[0]))

# Build matched pairs
image_paths = []
mask_paths = []
for img_name in image_files:
    if img_name in label_files:
        image_paths.append(os.path.join(image_dir, img_name))
        mask_paths.append(os.path.join(label_dir, img_name))

image_paths = np.array(image_paths)
mask_paths = np.array(mask_paths)

print(f'Total matched image-mask pairs: {len(image_paths)}')
print(f'Sample image path: {image_paths[0]}')
print(f'Sample mask path:  {mask_paths[0]}')

# ─── Visualize sample images ───
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Sample CT Images (top) and Masks (bottom)', fontsize=16, fontweight='bold')

sample_indices = np.random.choice(len(image_paths), 5, replace=False)
for i, idx in enumerate(sample_indices):
    img = Image.open(image_paths[idx]).convert('L')
    msk = Image.open(mask_paths[idx]).convert('L')

    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'Image {os.path.basename(image_paths[idx])}')
    axes[0, i].axis('off')

    axes[1, i].imshow(msk, cmap='gray')
    axes[1, i].set_title(f'Mask')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

# ─── Dataset statistics ───
sample_img = Image.open(image_paths[0]).convert('L')
sample_msk = Image.open(mask_paths[0]).convert('L')
print(f'\nOriginal image size: {sample_img.size}')
print(f'Original mask size:  {sample_msk.size}')
print(f'Image mode: {sample_img.mode}')
print(f'Mask unique values: {np.unique(np.array(sample_msk))}')

## 4. Loss Functions

Combined **Binary Cross-Entropy + Dice Loss** as described in the paper:
- BCE provides pixel-wise supervision and boundary precision
- Dice Loss handles class imbalance (stone pixels << background)

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss for binary segmentation."""

    def __init__(self, smooth=1e-7):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred_flat = pred.view(pred.size(0), -1)
        target_flat = target.view(target.size(0), -1)

        intersection = (pred_flat * target_flat).sum(dim=1)
        union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class BCEDiceLoss(nn.Module):
    """
    Combined Binary Cross-Entropy + Dice Loss.
    L_total = L_BCE + L_Dice
    """

    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        dice_loss = self.dice(pred, target)
        return bce_loss + dice_loss


# Quick test
criterion = BCEDiceLoss()
dummy_pred = torch.randn(2, 1, 256, 256)
dummy_target = torch.randint(0, 2, (2, 1, 256, 256)).float()
loss = criterion(dummy_pred, dummy_target)
print(f'Test loss: {loss.item():.4f}')

## 5. Evaluation Metrics

Compute: **Dice (DSC)**, **IoU**, **Accuracy**, **Precision**, **Recall**

In [ ]:
def compute_metrics(pred, target, threshold=0.5):
    """
    Compute segmentation metrics for a batch.

    Args:
        pred: model output logits (B, 1, H, W)
        target: ground truth masks (B, 1, H, W)
        threshold: binarization threshold

    Returns:
        dict with dice, iou, accuracy, precision, recall
    """
    pred_binary = (torch.sigmoid(pred) > threshold).float()
    target = target.float()

    # Flatten per sample
    pred_flat = pred_binary.view(pred_binary.size(0), -1)
    target_flat = target.view(target.size(0), -1)

    # True Positives, False Positives, False Negatives, True Negatives
    tp = (pred_flat * target_flat).sum(dim=1)
    fp = (pred_flat * (1 - target_flat)).sum(dim=1)
    fn = ((1 - pred_flat) * target_flat).sum(dim=1)
    tn = ((1 - pred_flat) * (1 - target_flat)).sum(dim=1)

    smooth = 1e-7

    # Dice Similarity Coefficient
    dice = (2.0 * tp + smooth) / (2.0 * tp + fp + fn + smooth)

    # Intersection over Union (Jaccard)
    iou = (tp + smooth) / (tp + fp + fn + smooth)

    # Pixel Accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    # Precision
    precision = (tp + smooth) / (tp + fp + smooth)

    # Recall (Sensitivity)
    recall = (tp + smooth) / (tp + fn + smooth)

    return {
        'dice': dice.mean().item(),
        'iou': iou.mean().item(),
        'accuracy': accuracy.mean().item(),
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
    }


# Quick test
metrics = compute_metrics(dummy_pred, dummy_target)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')

## 6. U-Net Architecture

Standard encoder-decoder with skip connections. Filter progression: 64 → 128 → 256 → 512 → 1024.

In [ ]:
class ConvBlock(nn.Module):
    """Two 3x3 Conv-BN-ReLU blocks (used across all architectures)."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """
    Standard U-Net for binary segmentation.
    Encoder: 4 downsampling blocks
    Bottleneck: 1 block at deepest level
    Decoder: 4 upsampling blocks with skip connections
    """

    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        # Encoder path
        for feat in features:
            self.encoders.append(ConvBlock(in_channels, feat))
            in_channels = feat

        # Bottleneck
        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)

        # Decoder path
        for feat in reversed(features):
            self.decoders.append(
                nn.ConvTranspose2d(feat * 2, feat, kernel_size=2, stride=2)
            )
            self.decoders.append(ConvBlock(feat * 2, feat))

        # Output
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder
        for encoder in self.encoders:
            x = encoder(x)
            skip_connections.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.decoders), 2):
            x = self.decoders[i](x)  # Upsample
            skip = skip_connections[i // 2]

            # Handle size mismatch
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)

            x = torch.cat([skip, x], dim=1)  # Concatenate skip connection
            x = self.decoders[i + 1](x)      # Conv block

        return self.final_conv(x)


# Test U-Net
model_test = UNet(in_channels=1, out_channels=1).to(device)
test_input = torch.randn(2, 1, 256, 256).to(device)
test_output = model_test(test_input)
print(f'U-Net | Input: {test_input.shape} → Output: {test_output.shape}')
total_params = sum(p.numel() for p in model_test.parameters())
print(f'U-Net | Total parameters: {total_params:,}')
del model_test, test_input, test_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 7. U-Net++ Architecture

Nested, dense skip connections between encoder and decoder. Each intermediate node aggregates features from all preceding nodes at the same level and the node below.

In [ ]:
class UNetPlusPlus(nn.Module):
    """
    U-Net++ with nested dense skip pathways.
    Uses deep supervision for multi-scale output.
    """

    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.depth = len(features)
        f = features

        # Encoder nodes X_{i,0}
        self.encoder0 = ConvBlock(in_channels, f[0])
        self.encoder1 = ConvBlock(f[0], f[1])
        self.encoder2 = ConvBlock(f[1], f[2])
        self.encoder3 = ConvBlock(f[2], f[3])

        # Bottleneck X_{4,0}
        self.bottleneck = ConvBlock(f[3], f[3] * 2)

        self.pool = nn.MaxPool2d(2, 2)

        # Nested dense skip pathway nodes X_{i,j}
        # Row 0 (shallowest)
        self.conv0_1 = ConvBlock(f[0] + f[1], f[0])
        self.conv0_2 = ConvBlock(f[0] * 2 + f[1], f[0])
        self.conv0_3 = ConvBlock(f[0] * 3 + f[1], f[0])
        self.conv0_4 = ConvBlock(f[0] * 4 + f[1], f[0])

        # Row 1
        self.conv1_1 = ConvBlock(f[1] + f[2], f[1])
        self.conv1_2 = ConvBlock(f[1] * 2 + f[2], f[1])
        self.conv1_3 = ConvBlock(f[1] * 3 + f[2], f[1])

        # Row 2
        self.conv2_1 = ConvBlock(f[2] + f[3], f[2])
        self.conv2_2 = ConvBlock(f[2] * 2 + f[3], f[2])

        # Row 3
        self.conv3_1 = ConvBlock(f[3] + f[3] * 2, f[3])

        # Upsampling layers
        self.up1_0 = nn.ConvTranspose2d(f[1], f[1], 2, stride=2)
        self.up2_0 = nn.ConvTranspose2d(f[2], f[2], 2, stride=2)
        self.up3_0 = nn.ConvTranspose2d(f[3], f[3], 2, stride=2)
        self.up4_0 = nn.ConvTranspose2d(f[3] * 2, f[3] * 2, 2, stride=2)

        self.up1_1 = nn.ConvTranspose2d(f[1], f[1], 2, stride=2)
        self.up2_1 = nn.ConvTranspose2d(f[2], f[2], 2, stride=2)
        self.up3_1 = nn.ConvTranspose2d(f[3], f[3], 2, stride=2)

        self.up1_2 = nn.ConvTranspose2d(f[1], f[1], 2, stride=2)
        self.up2_2 = nn.ConvTranspose2d(f[2], f[2], 2, stride=2)

        self.up1_3 = nn.ConvTranspose2d(f[1], f[1], 2, stride=2)

        # Deep supervision: output from X_{0,4}
        self.final_conv = nn.Conv2d(f[0], out_channels, 1)

    def _pad(self, x, target):
        """Pad x to match target spatial dimensions."""
        if x.shape[2:] != target.shape[2:]:
            x = F.interpolate(x, size=target.shape[2:], mode='bilinear', align_corners=True)
        return x

    def forward(self, x):
        # Encoder
        x0_0 = self.encoder0(x)
        x1_0 = self.encoder1(self.pool(x0_0))
        x2_0 = self.encoder2(self.pool(x1_0))
        x3_0 = self.encoder3(self.pool(x2_0))
        x4_0 = self.bottleneck(self.pool(x3_0))

        # Column 1
        x0_1 = self.conv0_1(torch.cat([x0_0, self._pad(self.up1_0(x1_0), x0_0)], dim=1))
        x1_1 = self.conv1_1(torch.cat([x1_0, self._pad(self.up2_0(x2_0), x1_0)], dim=1))
        x2_1 = self.conv2_1(torch.cat([x2_0, self._pad(self.up3_0(x3_0), x2_0)], dim=1))
        x3_1 = self.conv3_1(torch.cat([x3_0, self._pad(self.up4_0(x4_0), x3_0)], dim=1))

        # Column 2
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self._pad(self.up1_1(x1_1), x0_0)], dim=1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self._pad(self.up2_1(x2_1), x1_0)], dim=1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self._pad(self.up3_1(x3_1), x2_0)], dim=1))

        # Column 3
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self._pad(self.up1_2(x1_2), x0_0)], dim=1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self._pad(self.up2_2(x2_2), x1_0)], dim=1))

        # Column 4
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self._pad(self.up1_3(x1_3), x0_0)], dim=1))

        return self.final_conv(x0_4)


# Test U-Net++
model_test = UNetPlusPlus(in_channels=1, out_channels=1).to(device)
test_input = torch.randn(2, 1, 256, 256).to(device)
test_output = model_test(test_input)
print(f'U-Net++ | Input: {test_input.shape} → Output: {test_output.shape}')
total_params = sum(p.numel() for p in model_test.parameters())
print(f'U-Net++ | Total parameters: {total_params:,}')
del model_test, test_input, test_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 8. U-Net3+ Architecture

Full-scale skip connections: each decoder level aggregates feature maps from ALL encoder and decoder levels.

In [ ]:
class UNet3Plus(nn.Module):
    """
    U-Net3+ with full-scale skip connections.
    Each decoder level receives features from ALL encoder levels
    and all preceding decoder levels.
    """

    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        f = features
        cat_channels = f[0]  # Each connection contributes f[0] channels
        self.cat_blocks = 5  # 4 encoder levels + 1 bottleneck
        up_channels = cat_channels * self.cat_blocks  # Total channels after concat

        # Encoder
        self.encoder1 = ConvBlock(in_channels, f[0])
        self.encoder2 = ConvBlock(f[0], f[1])
        self.encoder3 = ConvBlock(f[1], f[2])
        self.encoder4 = ConvBlock(f[2], f[3])
        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = ConvBlock(f[3], f[3] * 2)

        # ─── Decoder 4 (receives from e1, e2, e3, e4, bottleneck) ───
        self.d4_e1 = nn.Sequential(
            nn.MaxPool2d(8, 8),
            nn.Conv2d(f[0], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d4_e2 = nn.Sequential(
            nn.MaxPool2d(4, 4),
            nn.Conv2d(f[1], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d4_e3 = nn.Sequential(
            nn.MaxPool2d(2, 2),
            nn.Conv2d(f[2], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d4_e4 = nn.Sequential(
            nn.Conv2d(f[3], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d4_bot = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(f[3] * 2, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d4_conv = ConvBlock(up_channels, up_channels)

        # ─── Decoder 3 ───
        self.d3_e1 = nn.Sequential(
            nn.MaxPool2d(4, 4),
            nn.Conv2d(f[0], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d3_e2 = nn.Sequential(
            nn.MaxPool2d(2, 2),
            nn.Conv2d(f[1], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d3_e3 = nn.Sequential(
            nn.Conv2d(f[2], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d3_d4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(up_channels, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d3_bot = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(f[3] * 2, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d3_conv = ConvBlock(up_channels, up_channels)

        # ─── Decoder 2 ───
        self.d2_e1 = nn.Sequential(
            nn.MaxPool2d(2, 2),
            nn.Conv2d(f[0], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d2_e2 = nn.Sequential(
            nn.Conv2d(f[1], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d2_d3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(up_channels, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d2_d4 = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(up_channels, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d2_bot = nn.Sequential(
            nn.Upsample(scale_factor=8, mode='bilinear', align_corners=True),
            nn.Conv2d(f[3] * 2, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d2_conv = ConvBlock(up_channels, up_channels)

        # ─── Decoder 1 ───
        self.d1_e1 = nn.Sequential(
            nn.Conv2d(f[0], cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d1_d2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(up_channels, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d1_d3 = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(up_channels, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d1_d4 = nn.Sequential(
            nn.Upsample(scale_factor=8, mode='bilinear', align_corners=True),
            nn.Conv2d(up_channels, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d1_bot = nn.Sequential(
            nn.Upsample(scale_factor=16, mode='bilinear', align_corners=True),
            nn.Conv2d(f[3] * 2, cat_channels, 1), nn.BatchNorm2d(cat_channels), nn.ReLU(True)
        )
        self.d1_conv = ConvBlock(up_channels, up_channels)

        # Output
        self.final_conv = nn.Conv2d(up_channels, out_channels, 1)

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)           # (B, 64, 256, 256)
        e2 = self.encoder2(self.pool(e1))  # (B, 128, 128, 128)
        e3 = self.encoder3(self.pool(e2))  # (B, 256, 64, 64)
        e4 = self.encoder4(self.pool(e3))  # (B, 512, 32, 32)
        bot = self.bottleneck(self.pool(e4))  # (B, 1024, 16, 16)

        # Decoder 4 (32x32)
        d4 = self.d4_conv(torch.cat([
            self.d4_e1(e1), self.d4_e2(e2), self.d4_e3(e3),
            self.d4_e4(e4), self.d4_bot(bot)
        ], dim=1))

        # Decoder 3 (64x64)
        d3 = self.d3_conv(torch.cat([
            self.d3_e1(e1), self.d3_e2(e2), self.d3_e3(e3),
            self.d3_d4(d4), self.d3_bot(bot)
        ], dim=1))

        # Decoder 2 (128x128)
        d2 = self.d2_conv(torch.cat([
            self.d2_e1(e1), self.d2_e2(e2),
            self.d2_d3(d3), self.d2_d4(d4), self.d2_bot(bot)
        ], dim=1))

        # Decoder 1 (256x256)
        d1 = self.d1_conv(torch.cat([
            self.d1_e1(e1), self.d1_d2(d2),
            self.d1_d3(d3), self.d1_d4(d4), self.d1_bot(bot)
        ], dim=1))

        return self.final_conv(d1)


# Test U-Net3+
model_test = UNet3Plus(in_channels=1, out_channels=1).to(device)
test_input = torch.randn(2, 1, 256, 256).to(device)
test_output = model_test(test_input)
print(f'U-Net3+ | Input: {test_input.shape} → Output: {test_output.shape}')
total_params = sum(p.numel() for p in model_test.parameters())
print(f'U-Net3+ | Total parameters: {total_params:,}')
del model_test, test_input, test_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 9. TransUNet Architecture

Hybrid CNN + Vision Transformer encoder with U-Net decoder. Captures both local spatial details (CNN) and long-range dependencies (Transformer).

In [ ]:
class PatchEmbedding(nn.Module):
    """Split feature maps into patches and project to embedding dimension."""

    def __init__(self, in_channels, embed_dim, patch_size=1):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H', W')
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, H'*W', embed_dim)
        x = self.norm(x)
        return x, H, W


class MultiHeadSelfAttention(nn.Module):
    """Multi-Head Self-Attention for Transformer."""

    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer encoder block with MHSA + MLP."""

    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TransUNet(nn.Module):
    """
    TransUNet: Transformer + U-Net hybrid.
    - CNN encoder extracts local features at multiple scales
    - Vision Transformer processes the deepest features for global context
    - U-Net decoder with skip connections reconstructs the segmentation mask
    """

    def __init__(self, in_channels=1, out_channels=1, img_size=256,
                 embed_dim=512, num_heads=8, num_layers=12,
                 features=[64, 128, 256, 512]):
        super().__init__()
        self.features = features

        # CNN Encoder (extract multi-scale features)
        self.encoder1 = ConvBlock(in_channels, features[0])  # 256x256
        self.encoder2 = ConvBlock(features[0], features[1])  # 128x128
        self.encoder3 = ConvBlock(features[1], features[2])  # 64x64
        self.encoder4 = ConvBlock(features[2], features[3])  # 32x32
        self.pool = nn.MaxPool2d(2, 2)

        # Patch embedding for Transformer
        # After encoder4 + pool: spatial size = img_size/16 = 16x16
        self.patch_embed = PatchEmbedding(features[3], embed_dim, patch_size=1)
        seq_len = (img_size // 16) ** 2  # 16*16 = 256 tokens
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Transformer encoder
        self.transformer = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
        self.transformer_norm = nn.LayerNorm(embed_dim)

        # Project transformer output back to CNN feature map
        self.proj_back = nn.Conv2d(embed_dim, features[3], 1)

        # Decoder with skip connections
        self.up4 = nn.ConvTranspose2d(features[3], features[3], 2, stride=2)
        self.dec4 = ConvBlock(features[3] + features[3], features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], 2, stride=2)
        self.dec3 = ConvBlock(features[2] + features[2], features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], 2, stride=2)
        self.dec2 = ConvBlock(features[1] + features[1], features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], 2, stride=2)
        self.dec1 = ConvBlock(features[0] + features[0], features[0])

        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        # CNN Encoder
        e1 = self.encoder1(x)              # (B, 64, 256, 256)
        e2 = self.encoder2(self.pool(e1))  # (B, 128, 128, 128)
        e3 = self.encoder3(self.pool(e2))  # (B, 256, 64, 64)
        e4 = self.encoder4(self.pool(e3))  # (B, 512, 32, 32)
        x = self.pool(e4)                  # (B, 512, 16, 16)

        # Patch Embedding + Transformer
        tokens, H, W = self.patch_embed(x)  # (B, 256, embed_dim)
        tokens = tokens + self.pos_embed
        tokens = self.transformer(tokens)
        tokens = self.transformer_norm(tokens)

        # Reshape back to spatial
        B, N, C = tokens.shape
        x = tokens.transpose(1, 2).reshape(B, C, H, W)  # (B, embed_dim, 16, 16)
        x = self.proj_back(x)  # (B, 512, 16, 16)

        # Decoder with skip connections
        x = self.up4(x)  # (B, 512, 32, 32)
        x = F.interpolate(x, size=e4.shape[2:], mode='bilinear', align_corners=True) if x.shape[2:] != e4.shape[2:] else x
        x = self.dec4(torch.cat([x, e4], dim=1))

        x = self.up3(x)  # (B, 256, 64, 64)
        x = F.interpolate(x, size=e3.shape[2:], mode='bilinear', align_corners=True) if x.shape[2:] != e3.shape[2:] else x
        x = self.dec3(torch.cat([x, e3], dim=1))

        x = self.up2(x)  # (B, 128, 128, 128)
        x = F.interpolate(x, size=e2.shape[2:], mode='bilinear', align_corners=True) if x.shape[2:] != e2.shape[2:] else x
        x = self.dec2(torch.cat([x, e2], dim=1))

        x = self.up1(x)  # (B, 64, 256, 256)
        x = F.interpolate(x, size=e1.shape[2:], mode='bilinear', align_corners=True) if x.shape[2:] != e1.shape[2:] else x
        x = self.dec1(torch.cat([x, e1], dim=1))

        return self.final_conv(x)


# Test TransUNet
model_test = TransUNet(in_channels=1, out_channels=1, img_size=256,
                       embed_dim=512, num_heads=8, num_layers=12).to(device)
test_input = torch.randn(2, 1, 256, 256).to(device)
test_output = model_test(test_input)
print(f'TransUNet | Input: {test_input.shape} → Output: {test_output.shape}')
total_params = sum(p.numel() for p in model_test.parameters())
print(f'TransUNet | Total parameters: {total_params:,}')
del model_test, test_input, test_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 10. Training Function

Generic training loop with validation, metric tracking, early stopping, and LR scheduling.

In [ ]:
def train_model(model, train_loader, val_loader, model_name='Model', fold=1):
    """
    Train a segmentation model with the paper's configuration.

    Returns:
        model: trained model
        history: dict with training/validation losses and metrics per epoch
    """
    model = model.to(device)
    criterion = BCEDiceLoss()
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'],
                          weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=CONFIG['lr_factor'],
        patience=CONFIG['lr_patience'], verbose=True
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'val_dice': [], 'val_iou': [],
        'val_accuracy': [], 'val_precision': [], 'val_recall': []
    }

    best_dice = 0.0
    patience_counter = 0
    best_model_state = None

    print(f'\n{"="*60}')
    print(f'Training {model_name} | Fold {fold}/{CONFIG["n_folds"]}')
    print(f'{"="*60}')

    for epoch in range(CONFIG['num_epochs']):
        # ─── Training Phase ───
        model.train()
        train_loss = 0.0

        for images, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{CONFIG["num_epochs"]} [Train]',
                                   leave=False):
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        # ─── Validation Phase ───
        model.eval()
        val_loss = 0.0
        val_metrics = defaultdict(float)
        num_val_batches = 0

        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f'Epoch {epoch+1}/{CONFIG["num_epochs"]} [Val]',
                                       leave=False):
                images, masks = images.to(device), masks.to(device)

                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item() * images.size(0)

                batch_metrics = compute_metrics(outputs, masks)
                for k, v in batch_metrics.items():
                    val_metrics[k] += v
                num_val_batches += 1

        val_loss /= len(val_loader.dataset)
        for k in val_metrics:
            val_metrics[k] /= num_val_batches

        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_metrics['dice'])
        history['val_iou'].append(val_metrics['iou'])
        history['val_accuracy'].append(val_metrics['accuracy'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])

        # LR Scheduler
        scheduler.step(val_metrics['dice'])

        # Logging
        if (epoch + 1) % 5 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'Epoch {epoch+1:3d} | '
                  f'Train Loss: {train_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | '
                  f'Dice: {val_metrics["dice"]:.4f} | '
                  f'IoU: {val_metrics["iou"]:.4f} | '
                  f'Acc: {val_metrics["accuracy"]:.4f} | '
                  f'LR: {current_lr:.6f}')

        # Early Stopping
        if val_metrics['dice'] > best_dice:
            best_dice = val_metrics['dice']
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= CONFIG['early_stop_patience']:
                print(f'\nEarly stopping at epoch {epoch+1}. Best Dice: {best_dice:.4f}')
                break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    print(f'\nBest Validation Dice: {best_dice:.4f}')

    return model, history

## 11. 5-Fold Cross-Validation Runner

Runs training for all 4 models across 5 folds, collecting metrics for comparison.

In [ ]:
def get_model(model_name):
    """Factory function to create a fresh model instance."""
    if model_name == 'U-Net':
        return UNet(in_channels=1, out_channels=1)
    elif model_name == 'U-Net++':
        return UNetPlusPlus(in_channels=1, out_channels=1)
    elif model_name == 'U-Net3+':
        return UNet3Plus(in_channels=1, out_channels=1)
    elif model_name == 'TransUNet':
        return TransUNet(in_channels=1, out_channels=1, img_size=CONFIG['img_size'],
                         embed_dim=512, num_heads=8, num_layers=12)
    else:
        raise ValueError(f'Unknown model: {model_name}')


def run_cross_validation(model_names=None):
    """
    Run 5-fold cross-validation for specified models.

    Args:
        model_names: list of model names to evaluate.
                     Default: all four architectures.

    Returns:
        all_results: dict mapping model_name -> list of per-fold metric dicts
        all_histories: dict mapping model_name -> list of per-fold training histories
    """
    if model_names is None:
        model_names = ['U-Net', 'U-Net++', 'U-Net3+', 'TransUNet']

    kfold = KFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])

    all_results = {}
    all_histories = {}

    for model_name in model_names:
        print(f'\n{"#"*70}')
        print(f'  MODEL: {model_name}')
        print(f'{"#"*70}')

        fold_results = []
        fold_histories = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(image_paths), 1):
            # Create datasets
            train_dataset = KidneyStoneDataset(
                image_paths[train_idx], mask_paths[train_idx],
                img_size=CONFIG['img_size'], augment=True
            )
            val_dataset = KidneyStoneDataset(
                image_paths[val_idx], mask_paths[val_idx],
                img_size=CONFIG['img_size'], augment=False
            )

            train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                                      shuffle=True, num_workers=0, pin_memory=True)
            val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                                    shuffle=False, num_workers=0, pin_memory=True)

            print(f'\nFold {fold}: Train={len(train_dataset)}, Val={len(val_dataset)}')

            # Create fresh model
            model = get_model(model_name)

            # Train
            model, history = train_model(model, train_loader, val_loader,
                                          model_name=model_name, fold=fold)

            # Final evaluation on validation set
            model.eval()
            final_metrics = defaultdict(float)
            num_batches = 0
            with torch.no_grad():
                for images, masks in val_loader:
                    images, masks = images.to(device), masks.to(device)
                    outputs = model(images)
                    batch_metrics = compute_metrics(outputs, masks)
                    for k, v in batch_metrics.items():
                        final_metrics[k] += v
                    num_batches += 1

            for k in final_metrics:
                final_metrics[k] /= num_batches

            fold_results.append(dict(final_metrics))
            fold_histories.append(history)

            print(f'Fold {fold} Final → Dice: {final_metrics["dice"]:.4f}, '
                  f'IoU: {final_metrics["iou"]:.4f}, Acc: {final_metrics["accuracy"]:.4f}')

            # Cleanup
            del model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

        all_results[model_name] = fold_results
        all_histories[model_name] = fold_histories

    return all_results, all_histories


# ╔══════════════════════════════════════════════════════════╗
# ║  RUN CROSS-VALIDATION                                    ║
# ║  Uncomment to train all models, or select specific ones  ║
# ╚══════════════════════════════════════════════════════════╝

# Train ALL four models (full replication):
all_results, all_histories = run_cross_validation()

# Or train specific models:
# all_results, all_histories = run_cross_validation(['U-Net'])
# all_results, all_histories = run_cross_validation(['U-Net', 'U-Net++'])

## 12. Results Comparison

Tabular and graphical comparison of all models across folds.

In [ ]:
def print_results_table(all_results):
    """
    Print a formatted comparison table of all models.
    Shows mean ± std across 5 folds for each metric.
    """
    metrics = ['dice', 'iou', 'accuracy', 'precision', 'recall']
    metric_names = ['Dice (%)', 'IoU (%)', 'Accuracy (%)', 'Precision (%)', 'Recall (%)']

    print(f'\n{"="*90}')
    print(f'{"5-Fold Cross-Validation Results":^90}')
    print(f'{"="*90}')
    print(f'{"Model":<15}', end='')
    for name in metric_names:
        print(f'{name:>15}', end='')
    print()
    print(f'{"-"*90}')

    for model_name, fold_results in all_results.items():
        print(f'{model_name:<15}', end='')
        for metric in metrics:
            values = [r[metric] * 100 for r in fold_results]
            mean = np.mean(values)
            std = np.std(values)
            print(f'{mean:>10.2f}±{std:<4.2f}', end='')
        print()
    print(f'{"="*90}')


def plot_training_curves(all_histories):
    """Plot training/validation loss and Dice curves for each model."""
    n_models = len(all_histories)
    fig, axes = plt.subplots(n_models, 2, figsize=(16, 5 * n_models))

    if n_models == 1:
        axes = axes.reshape(1, -1)

    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

    for i, (model_name, histories) in enumerate(all_histories.items()):
        # Loss curves
        for fold_idx, hist in enumerate(histories):
            axes[i, 0].plot(hist['train_loss'], alpha=0.3, color=colors[fold_idx],
                            label=f'Fold {fold_idx+1} Train')
            axes[i, 0].plot(hist['val_loss'], alpha=0.8, color=colors[fold_idx],
                            linestyle='--', label=f'Fold {fold_idx+1} Val')

        axes[i, 0].set_title(f'{model_name} — Loss', fontsize=14, fontweight='bold')
        axes[i, 0].set_xlabel('Epoch')
        axes[i, 0].set_ylabel('Loss')
        axes[i, 0].legend(fontsize=8, ncol=2)
        axes[i, 0].grid(True, alpha=0.3)

        # Dice curves
        for fold_idx, hist in enumerate(histories):
            axes[i, 1].plot(hist['val_dice'], alpha=0.8, color=colors[fold_idx],
                            label=f'Fold {fold_idx+1}')

        axes[i, 1].set_title(f'{model_name} — Validation Dice', fontsize=14, fontweight='bold')
        axes[i, 1].set_xlabel('Epoch')
        axes[i, 1].set_ylabel('Dice Score')
        axes[i, 1].legend(fontsize=8)
        axes[i, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_model_comparison(all_results):
    """Bar chart comparing all models across metrics."""
    metrics = ['dice', 'iou', 'accuracy', 'precision', 'recall']
    metric_labels = ['Dice', 'IoU', 'Accuracy', 'Precision', 'Recall']

    model_names = list(all_results.keys())
    x = np.arange(len(metrics))
    width = 0.18
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

    fig, ax = plt.subplots(figsize=(14, 7))

    for i, model_name in enumerate(model_names):
        means = [np.mean([r[m] * 100 for r in all_results[model_name]]) for m in metrics]
        stds = [np.std([r[m] * 100 for r in all_results[model_name]]) for m in metrics]
        bars = ax.bar(x + i * width, means, width, label=model_name,
                      color=colors[i % len(colors)], alpha=0.85,
                      yerr=stds, capsize=3)

    ax.set_ylabel('Score (%)', fontsize=13)
    ax.set_title('Model Comparison — 5-Fold Cross-Validation', fontsize=16, fontweight='bold')
    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels(metric_labels, fontsize=12)
    ax.legend(fontsize=11)
    ax.set_ylim([80, 102])
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


# ─── Display Results ───
print_results_table(all_results)
plot_training_curves(all_histories)
plot_model_comparison(all_results)

## 13. Qualitative Results

Visualize predictions from each model alongside ground truth.

In [ ]:
def visualize_predictions(all_results, all_histories, num_samples=5):
    """
    Visualize model predictions vs ground truth.
    Uses the last fold's trained model for each architecture.
    """
    model_names = list(all_results.keys())

    # Create a validation dataset from the last fold
    kfold = KFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])
    folds = list(kfold.split(image_paths))
    _, val_idx = folds[-1]

    val_dataset = KidneyStoneDataset(
        image_paths[val_idx], mask_paths[val_idx],
        img_size=CONFIG['img_size'], augment=False
    )

    # Select random samples
    sample_indices = np.random.choice(len(val_dataset), min(num_samples, len(val_dataset)), replace=False)

    n_cols = 2 + len(model_names)  # image + GT + each model
    fig, axes = plt.subplots(num_samples, n_cols, figsize=(4 * n_cols, 4 * num_samples))

    # Column headers
    col_titles = ['CT Image', 'Ground Truth'] + model_names
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=13, fontweight='bold')

    for row, idx in enumerate(sample_indices):
        image, mask = val_dataset[idx]

        # Display CT image
        axes[row, 0].imshow(image.squeeze().numpy(), cmap='gray')
        axes[row, 0].axis('off')

        # Display ground truth
        axes[row, 1].imshow(mask.squeeze().numpy(), cmap='gray')
        axes[row, 1].axis('off')

        # Generate and display predictions from each model
        for col, model_name in enumerate(model_names, start=2):
            # Re-create and load the best model from the last fold
            model = get_model(model_name).to(device)
            model.eval()

            with torch.no_grad():
                pred = model(image.unsqueeze(0).to(device))
                pred_mask = (torch.sigmoid(pred) > 0.5).float().cpu().squeeze().numpy()

            axes[row, col].imshow(pred_mask, cmap='gray')
            axes[row, col].axis('off')

            del model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

    plt.suptitle('Qualitative Comparison: Predictions vs Ground Truth',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('qualitative_results.png', dpi=150, bbox_inches='tight')
    plt.show()


# ─── Note ───
# For proper qualitative evaluation, you should save the best model weights
# during training and reload them here. The visualization below uses
# untrained models as a placeholder to demonstrate the visualization pipeline.
# During actual training (Cell 11), the models will be trained and evaluated.
print('Qualitative visualization pipeline ready.')
print('After running the full cross-validation (Cell 11), modify this cell')
print('to load the saved best model weights for proper visualization.')

## 14. Save Results

Save the quantitative results to a CSV file for reporting.

In [ ]:
def save_results_csv(all_results, filename='results_5fold_cv.csv'):
    """Save cross-validation results to CSV."""
    metrics = ['dice', 'iou', 'accuracy', 'precision', 'recall']

    lines = ['Model,Fold,Dice,IoU,Accuracy,Precision,Recall']

    for model_name, fold_results in all_results.items():
        for fold_idx, result in enumerate(fold_results, 1):
            values = [f'{result[m]*100:.4f}' for m in metrics]
            lines.append(f'{model_name},{fold_idx},{",".join(values)}')

        # Add mean row
        means = [f'{np.mean([r[m]*100 for r in fold_results]):.4f}' for m in metrics]
        lines.append(f'{model_name},Mean,{",".join(means)}')

        # Add std row
        stds = [f'{np.std([r[m]*100 for r in fold_results]):.4f}' for m in metrics]
        lines.append(f'{model_name},Std,{",".join(stds)}')

    with open(filename, 'w') as f:
        f.write('\n'.join(lines))

    print(f'Results saved to {filename}')


# Save results after training
try:
    save_results_csv(all_results)
except NameError:
    print('Run cross-validation first (Cell 11) to generate results.')